# S05 ? Data engineering AWS stack

This notebook deploys one parent CloudFormation stack with separate templates for IAM roles, Kinesis and Firehose, MySQL RDS, EMR, and provisioned MSK. Run the base deployment first. Run the EMR step only when you need the cluster; it can take much longer.

The child templates are defined inside this notebook. The notebook uploads them to the data lake bucket using content based object names. Updating one child template leaves the others unchanged. Each run still incurs charges for the resources it creates. Delete the stack when finished.

RDS and EMR use the same **default VPC**. RDS receives default subnets in at least two Availability Zones. The allowed client range uses your public IP's first three octets (`/24`), for both MySQL and SSH. That range includes up to 256 addresses, so use it only for this lab.


## 1. Settings and AWS session

Set a stable stack name so later runs update the same stack. The named AWS profile and EC2 key pair must already exist. The bucket must exist and be accessible. Set `MYSQL_USERNAME` and `MYSQL_PASSWORD` in the first code cell; their defaults are `mysqladmin` and `admin1234`.


In [ ]:
from pathlib import Path
import boto3
from botocore.exceptions import ClientError

AWS_PROFILE = "training"
AWS_REGION = "us-east-1"
MYSQL_USERNAME = "mysqladmin"
MYSQL_PASSWORD = "admin1234"
MSK_CLUSTER_NAME = "dataeng-training-kafka"
MSK_BROKER_STORAGE_GB = 10
MSK_KAFKA_VERSION = "3.9.x"
STACK_NAME = "dataeng-training"
S3_DATALAKE = "gks-datalake"
CLUSTER_NAME = "My cluster 1"
KINESIS_STREAM_NAME = "gks-dataeng-invoices"
KINESIS_FIREHOSE_NAME = "gks-dataeng-invoices-to-s3"
KEY_NAME = "ec2emrkey"
RELEASE_LABEL = "emr-7.14.0"
INSTANCE_TYPE = "m4.large"
CORE_INSTANCE_COUNT = 1
EBS_DATA_VOLUME_SIZE_GB = 32
EBS_ROOT_VOLUME_SIZE_GB = 30
IDLE_TIMEOUT_SECONDS = 36000

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
print("Account:", session.client("sts").get_caller_identity()["Account"])
s3 = session.client("s3")
ec2 = session.client("ec2")
cloudformation = session.client("cloudformation")
s3.head_bucket(Bucket=S3_DATALAKE)
print("Bucket:", S3_DATALAKE)


## CloudFormation templates

All five child templates and the parent template are defined below. Edit the YAML here if you want to change resources. The notebook uploads their text to S3 for CloudFormation nested stacks.


In [ ]:
TEMPLATE_NAMES = {'RolesTemplateURL': 'emr_roles.yaml', 'PipelineTemplateURL': 'invoice_pipeline.yaml', 'RDSTemplateURL': 'mysql_rds.yaml', 'EMRTemplateURL': 'emr_cluster.yaml', 'MSKTemplateURL': 'msk_kafka.yaml'}

TEMPLATE_BODIES = {
    'RolesTemplateURL': """AWSTemplateFormatVersion: '2010-09-09'
Description: IAM roles and instance profile for the training EMR cluster.
Parameters:
  NamePrefix:
    Type: String
Resources:
  EMRServiceRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: !Sub '${NamePrefix}-service-role'
      Path: /service-role/
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal: {Service: elasticmapreduce.amazonaws.com}
            Action: sts:AssumeRole
      ManagedPolicyArns:
        - arn:aws:iam::aws:policy/service-role/AmazonEMRServicePolicy_v2
      Policies:
        - PolicyName: PassClusterEC2Role
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action: iam:PassRole
                Resource: !Sub 'arn:${AWS::Partition}:iam::${AWS::AccountId}:role/${NamePrefix}-ec2-role'
                Condition:
                  StringLike:
                    iam:PassedToService: ec2.amazonaws.com
      Tags:
        - {Key: for-use-with-amazon-emr-managed-policies, Value: 'true'}
  EMREC2Role:
    Type: AWS::IAM::Role
    Properties:
      RoleName: !Sub '${NamePrefix}-ec2-role'
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal: {Service: ec2.amazonaws.com}
            Action: sts:AssumeRole
      ManagedPolicyArns:
        - arn:aws:iam::aws:policy/AmazonS3FullAccess
        - arn:aws:iam::aws:policy/AWSGlueConsoleFullAccess
        - arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy
      Tags:
        - {Key: for-use-with-amazon-emr-managed-policies, Value: 'true'}
  EMREC2InstanceProfile:
    Type: AWS::IAM::InstanceProfile
    Properties:
      InstanceProfileName: !Sub '${NamePrefix}-instance-profile'
      Roles: [!Ref EMREC2Role]
Outputs:
  ServiceRoleArn:
    Value: !GetAtt EMRServiceRole.Arn
  InstanceProfileName:
    Value: !Ref EMREC2InstanceProfile
""",
    'PipelineTemplateURL': """AWSTemplateFormatVersion: '2010-09-09'
Description: Kinesis stream and Firehose delivery to S3.
Parameters:
  BucketName: {Type: String}
  StreamName: {Type: String}
  FirehoseName: {Type: String}
Resources:
  InvoiceStream:
    Type: AWS::Kinesis::Stream
    Properties:
      Name: !Ref StreamName
      ShardCount: 1
      StreamModeDetails: {StreamMode: PROVISIONED}
  FirehoseDeliveryRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal: {Service: firehose.amazonaws.com}
            Action: sts:AssumeRole
      Policies:
        - PolicyName: ReadInvoicesAndWriteS3
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action: [kinesis:DescribeStream, kinesis:GetShardIterator, kinesis:GetRecords, kinesis:ListShards]
                Resource: !GetAtt InvoiceStream.Arn
              - Effect: Allow
                Action: [s3:GetBucketLocation, s3:ListBucket]
                Resource: !Sub 'arn:${AWS::Partition}:s3:::${BucketName}'
              - Effect: Allow
                Action: [s3:PutObject, s3:AbortMultipartUpload, s3:ListMultipartUploadParts]
                Resource:
                  - !Sub 'arn:${AWS::Partition}:s3:::${BucketName}/kinesis/invoices/*'
                  - !Sub 'arn:${AWS::Partition}:s3:::${BucketName}/kinesis/invoice-errors/*'
  InvoiceFirehose:
    Type: AWS::KinesisFirehose::DeliveryStream
    Properties:
      DeliveryStreamName: !Ref FirehoseName
      DeliveryStreamType: KinesisStreamAsSource
      KinesisStreamSourceConfiguration:
        KinesisStreamARN: !GetAtt InvoiceStream.Arn
        RoleARN: !GetAtt FirehoseDeliveryRole.Arn
      ExtendedS3DestinationConfiguration:
        BucketARN: !Sub 'arn:${AWS::Partition}:s3:::${BucketName}'
        RoleARN: !GetAtt FirehoseDeliveryRole.Arn
        Prefix: 'kinesis/invoices/!{timestamp:yyyy/MM/dd}/'
        ErrorOutputPrefix: 'kinesis/invoice-errors/!{firehose:error-output-type}/!{timestamp:yyyy/MM/dd}/'
        BufferingHints: {IntervalInSeconds: 300, SizeInMBs: 1}
Outputs:
  KinesisStreamName:
    Value: !Ref InvoiceStream
  FirehoseDeliveryStreamName:
    Value: !Ref InvoiceFirehose
""",
    'RDSTemplateURL': """AWSTemplateFormatVersion: '2010-09-09'
Description: MySQL RDS instance in the default VPC.
Parameters:
  VpcId: {Type: AWS::EC2::VPC::Id}
  SubnetIds: {Type: 'List<AWS::EC2::Subnet::Id>'}
  AllowedCidr: {Type: String}
  EMRSecurityGroupId: {Type: AWS::EC2::SecurityGroup::Id}
  DBUsername: {Type: String, Default: mysqladmin}
  DBPassword: {Type: String, NoEcho: true, MinLength: 8}
Resources:
  DatabaseSecurityGroup:
    Type: AWS::EC2::SecurityGroup
    Properties:
      GroupDescription: MySQL access from the training network
      VpcId: !Ref VpcId
      SecurityGroupIngress:
        - IpProtocol: tcp
          FromPort: 3306
          ToPort: 3306
          CidrIp: !Ref AllowedCidr
        - IpProtocol: tcp
          FromPort: 3306
          ToPort: 3306
          SourceSecurityGroupId: !Ref EMRSecurityGroupId
  DatabaseSubnetGroup:
    Type: AWS::RDS::DBSubnetGroup
    Properties:
      DBSubnetGroupDescription: Default VPC subnets for training MySQL
      SubnetIds: !Ref SubnetIds
  Database:
    Type: AWS::RDS::DBInstance
    DeletionPolicy: Delete
    Properties:
      Engine: mysql
      DBInstanceClass: db.t3.micro
      AllocatedStorage: '20'
      StorageType: gp3
      PubliclyAccessible: true
      MultiAZ: false
      BackupRetentionPeriod: 0
      MasterUsername: !Ref DBUsername
      MasterUserPassword: !Ref DBPassword
      DBSubnetGroupName: !Ref DatabaseSubnetGroup
      VPCSecurityGroups: [!Ref DatabaseSecurityGroup]
Outputs:
  Endpoint:
    Value: !GetAtt Database.Endpoint.Address
  Port:
    Value: !GetAtt Database.Endpoint.Port
""",
    'EMRTemplateURL': """AWSTemplateFormatVersion: '2010-09-09'
Description: Training EMR cluster, deployed independently of pipeline and database resources.
Parameters:
  ClusterName: {Type: String}
  BucketName: {Type: String}
  ServiceRoleArn: {Type: String}
  InstanceProfileName: {Type: String}
  SubnetId: {Type: AWS::EC2::Subnet::Id}
  SecurityGroupId: {Type: AWS::EC2::SecurityGroup::Id}
  KeyName: {Type: AWS::EC2::KeyPair::KeyName}
  ReleaseLabel: {Type: String}
  InstanceType: {Type: String}
  CoreInstanceCount: {Type: Number, Default: 1}
  DataVolumeSizeGB: {Type: Number, Default: 32}
  RootVolumeSizeGB: {Type: Number, Default: 30}
  IdleTimeoutSeconds: {Type: Number, Default: 36000}
Resources:
  EMRCluster:
    Type: AWS::EMR::Cluster
    Properties:
      Name: !Ref ClusterName
      ReleaseLabel: !Ref ReleaseLabel
      LogUri: !Sub 's3://${BucketName}/emr-logs/'
      ServiceRole: !Ref ServiceRoleArn
      JobFlowRole: !Ref InstanceProfileName
      VisibleToAllUsers: true
      EbsRootVolumeSize: !Ref RootVolumeSizeGB
      ScaleDownBehavior: TERMINATE_AT_TASK_COMPLETION
      AutoTerminationPolicy:
        IdleTimeout: !Ref IdleTimeoutSeconds
      Applications:
        - {Name: AmazonCloudWatchAgent}
        - {Name: Hadoop}
        - {Name: Hive}
        - {Name: Hue}
        - {Name: JupyterEnterpriseGateway}
        - {Name: JupyterHub}
        - {Name: Livy}
        - {Name: Spark}
        - {Name: Trino}
      Configurations:
        - Classification: spark-hive-site
          ConfigurationProperties:
            hive.metastore.client.factory.class: com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory
      Instances:
        Ec2KeyName: !Ref KeyName
        Ec2SubnetId: !Ref SubnetId
        EmrManagedMasterSecurityGroup: !Ref SecurityGroupId
        EmrManagedSlaveSecurityGroup: !Ref SecurityGroupId
        KeepJobFlowAliveWhenNoSteps: true
        TerminationProtected: false
        UnhealthyNodeReplacement: true
        MasterInstanceGroup:
          InstanceCount: 1
          InstanceType: !Ref InstanceType
          Market: ON_DEMAND
          Name: Primary
          EbsConfiguration:
            EbsBlockDeviceConfigs:
              - VolumeSpecification:
                  VolumeType: gp2
                  SizeInGB: !Ref DataVolumeSizeGB
                VolumesPerInstance: 1
        CoreInstanceGroup:
          InstanceCount: !Ref CoreInstanceCount
          InstanceType: !Ref InstanceType
          Market: ON_DEMAND
          Name: Core
          EbsConfiguration:
            EbsBlockDeviceConfigs:
              - VolumeSpecification:
                  VolumeType: gp2
                  SizeInGB: !Ref DataVolumeSizeGB
                VolumesPerInstance: 1
      Tags:
        - {Key: for-use-with-amazon-emr-managed-policies, Value: 'true'}
        - {Key: Purpose, Value: data-engineering-training}
Outputs:
  ClusterId:
    Value: !Ref EMRCluster
  ClusterPrimaryPublicDns:
    Value: !GetAtt EMRCluster.MasterPublicDNS
""",
    'MSKTemplateURL': """AWSTemplateFormatVersion: '2010-09-09'
Description: Minimal provisioned MSK cluster in two default VPC subnets.
Parameters:
  ClusterName: {Type: String}
  VpcId: {Type: AWS::EC2::VPC::Id}
  VpcCidr: {Type: String}
  SubnetIds: {Type: 'List<AWS::EC2::Subnet::Id>'}
  BrokerStorageGB: {Type: Number, Default: 10, MinValue: 10, MaxValue: 20}
  KafkaVersion: {Type: String, Default: '3.9.x'}
Resources:
  KafkaSecurityGroup:
    Type: AWS::EC2::SecurityGroup
    Properties:
      GroupDescription: TLS Kafka access from the default VPC
      VpcId: !Ref VpcId
      SecurityGroupIngress:
        - IpProtocol: tcp
          FromPort: 9094
          ToPort: 9094
          CidrIp: !Ref VpcCidr
  BrokerInternalIngress:
    Type: AWS::EC2::SecurityGroupIngress
    Properties:
      GroupId: !Ref KafkaSecurityGroup
      IpProtocol: '-1'
      SourceSecurityGroupId: !Ref KafkaSecurityGroup
      Description: Broker communication within the cluster
  KafkaCluster:
    Type: AWS::MSK::Cluster
    DependsOn: BrokerInternalIngress
    Properties:
      ClusterName: !Ref ClusterName
      KafkaVersion: !Ref KafkaVersion
      NumberOfBrokerNodes: 2
      StorageMode: LOCAL
      BrokerNodeGroupInfo:
        InstanceType: kafka.t3.small
        ClientSubnets: !Ref SubnetIds
        SecurityGroups: [!Ref KafkaSecurityGroup]
        StorageInfo:
          EBSStorageInfo:
            VolumeSize: !Ref BrokerStorageGB
      EncryptionInfo:
        EncryptionInTransit:
          ClientBroker: TLS
          InCluster: true
Outputs:
  ClusterArn:
    Value: !Ref KafkaCluster
""",
}

PARENT_TEMPLATE = """AWSTemplateFormatVersion: '2010-09-09'
Description: Parent stack for training IAM, invoice pipeline, MySQL, and optional EMR.
Parameters:
  RolesTemplateURL: {Type: String}
  PipelineTemplateURL: {Type: String}
  RDSTemplateURL: {Type: String}
  EMRTemplateURL: {Type: String}
  MSKTemplateURL: {Type: String}
  DeployEMR: {Type: String, Default: 'false', AllowedValues: ['true', 'false']}
  DeployMSK: {Type: String, Default: 'false', AllowedValues: ['true', 'false']}
  BucketName: {Type: String}
  StreamName: {Type: String}
  FirehoseName: {Type: String}
  VpcId: {Type: AWS::EC2::VPC::Id}
  SubnetIds: {Type: CommaDelimitedList}
  MSKSubnetIds: {Type: CommaDelimitedList}
  VpcCidr: {Type: String}
  EMRSubnetId: {Type: AWS::EC2::Subnet::Id}
  EMRSecurityGroupId: {Type: AWS::EC2::SecurityGroup::Id}
  AllowedCidr: {Type: String}
  DBUsername: {Type: String, Default: mysqladmin}
  DBPassword: {Type: String, NoEcho: true, MinLength: 8}
  ClusterName: {Type: String}
  KeyName: {Type: AWS::EC2::KeyPair::KeyName}
  ReleaseLabel: {Type: String}
  InstanceType: {Type: String}
  CoreInstanceCount: {Type: Number, Default: 1}
  DataVolumeSizeGB: {Type: Number, Default: 32}
  RootVolumeSizeGB: {Type: Number, Default: 30}
  IdleTimeoutSeconds: {Type: Number, Default: 36000}
  MSKClusterName: {Type: String}
  MSKBrokerStorageGB: {Type: Number, Default: 10, MinValue: 10, MaxValue: 20}
  MSKKafkaVersion: {Type: String, Default: '3.9.x'}
Conditions:
  CreateEMR: !Equals [!Ref DeployEMR, 'true']
  CreateMSK: !Equals [!Ref DeployMSK, 'true']
Resources:
  Roles:
    Type: AWS::CloudFormation::Stack
    Properties:
      TemplateURL: !Ref RolesTemplateURL
      Parameters:
        NamePrefix: !Ref AWS::StackName
  Pipeline:
    Type: AWS::CloudFormation::Stack
    Properties:
      TemplateURL: !Ref PipelineTemplateURL
      Parameters:
        BucketName: !Ref BucketName
        StreamName: !Ref StreamName
        FirehoseName: !Ref FirehoseName
  RDS:
    Type: AWS::CloudFormation::Stack
    Properties:
      TemplateURL: !Ref RDSTemplateURL
      Parameters:
        VpcId: !Ref VpcId
        SubnetIds: !Join [',', !Ref SubnetIds]
        AllowedCidr: !Ref AllowedCidr
        EMRSecurityGroupId: !Ref EMRSecurityGroupId
        DBUsername: !Ref DBUsername
        DBPassword: !Ref DBPassword
  EMR:
    Type: AWS::CloudFormation::Stack
    Condition: CreateEMR
    Properties:
      TemplateURL: !Ref EMRTemplateURL
      Parameters:
        ClusterName: !Ref ClusterName
        BucketName: !Ref BucketName
        ServiceRoleArn: !GetAtt Roles.Outputs.ServiceRoleArn
        InstanceProfileName: !GetAtt Roles.Outputs.InstanceProfileName
        SubnetId: !Ref EMRSubnetId
        SecurityGroupId: !Ref EMRSecurityGroupId
        KeyName: !Ref KeyName
        ReleaseLabel: !Ref ReleaseLabel
        InstanceType: !Ref InstanceType
        CoreInstanceCount: !Ref CoreInstanceCount
        DataVolumeSizeGB: !Ref DataVolumeSizeGB
        RootVolumeSizeGB: !Ref RootVolumeSizeGB
        IdleTimeoutSeconds: !Ref IdleTimeoutSeconds
  MSK:
    Type: AWS::CloudFormation::Stack
    Condition: CreateMSK
    Properties:
      TemplateURL: !Ref MSKTemplateURL
      Parameters:
        ClusterName: !Ref MSKClusterName
        VpcId: !Ref VpcId
        VpcCidr: !Ref VpcCidr
        SubnetIds: !Join [',', !Ref MSKSubnetIds]
        BrokerStorageGB: !Ref MSKBrokerStorageGB
        KafkaVersion: !Ref MSKKafkaVersion
Outputs:
  KinesisStreamName:
    Value: !GetAtt Pipeline.Outputs.KinesisStreamName
  FirehoseDeliveryStreamName:
    Value: !GetAtt Pipeline.Outputs.FirehoseDeliveryStreamName
  RDSEndpoint:
    Value: !GetAtt RDS.Outputs.Endpoint
  RDSPort:
    Value: !GetAtt RDS.Outputs.Port
  EMRClusterId:
    Condition: CreateEMR
    Value: !GetAtt EMR.Outputs.ClusterId
  MSKClusterArn:
    Condition: CreateMSK
    Value: !GetAtt MSK.Outputs.ClusterArn
"""


## Deployment helpers

These functions discover the default VPC, create the client network rule, upload the templates, and deploy the parent stack.


In [ ]:
from hashlib import sha256
from ipaddress import IPv4Address, IPv4Network
from botocore.exceptions import WaiterError

def default_network(ec2, key_name):
    vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"]
    if len(vpcs) != 1:
        raise RuntimeError("Expected one default VPC in this region")
    vpc_id = vpcs[0]["VpcId"]
    subnets = ec2.describe_subnets(Filters=[
        {"Name": "vpc-id", "Values": [vpc_id]},
        {"Name": "default-for-az", "Values": ["true"]},
        {"Name": "state", "Values": ["available"]},
    ])["Subnets"]
    by_az = {}
    for subnet in sorted(subnets, key=lambda s: s.get("AvailableIpAddressCount", 0), reverse=True):
        by_az.setdefault(subnet["AvailabilityZone"], subnet)
    if len(by_az) < 2:
        raise RuntimeError("RDS needs default VPC subnets in at least two Availability Zones")
    selected = list(by_az.values())
    msk_subnets = [s for s in selected if s.get("AvailabilityZoneId") != "use1-az3"][:2]
    if len(msk_subnets) != 2:
        raise RuntimeError("MSK needs two default VPC subnets outside use1-az3")
    emr_subnet = selected[0]
    if emr_subnet.get("AvailableIpAddressCount", 0) < 10:
        raise RuntimeError("The selected EMR subnet has fewer than 10 free IPv4 addresses")
    groups = ec2.describe_security_groups(Filters=[
        {"Name": "vpc-id", "Values": [vpc_id]},
        {"Name": "group-name", "Values": ["default"]},
    ])["SecurityGroups"]
    if len(groups) != 1:
        raise RuntimeError("Default security group not found")
    ec2.describe_key_pairs(KeyNames=[key_name])
    return {
        "VpcId": vpc_id,
        "SubnetIds": [s["SubnetId"] for s in selected],
        "MSKSubnetIds": [s["SubnetId"] for s in msk_subnets],
        "VpcCidr": vpcs[0]["CidrBlock"],
        "EMRSubnetId": emr_subnet["SubnetId"],
        "EMRSecurityGroupId": groups[0]["GroupId"],
    }


def first_three_octets_cidr(public_ip):
    ip = IPv4Address(public_ip.strip())
    return str(IPv4Network(f"{ip}/24", strict=False))


def allow_ssh(ec2, security_group_id, cidr):
    try:
        ec2.authorize_security_group_ingress(
            GroupId=security_group_id,
            IpPermissions=[{
                "IpProtocol": "tcp", "FromPort": 22, "ToPort": 22,
                "IpRanges": [{"CidrIp": cidr, "Description": "Training network SSH access"}],
            }],
        )
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "InvalidPermission.Duplicate":
            raise


def upload_templates(s3, bucket, stack_name, region):
    """Content-based keys ensure unchanged nested stacks keep the same TemplateURL."""
    urls = {}
    for parameter, filename in TEMPLATE_NAMES.items():
        body = TEMPLATE_BODIES[parameter].encode('utf-8')
        digest = sha256(body).hexdigest()[:16]
        key = f"cloudformation/{stack_name}/{filename.removesuffix('.yaml')}-{digest}.yaml"
        s3.put_object(Bucket=bucket, Key=key, Body=body, ContentType="text/yaml")
        urls[parameter] = f"https://{bucket}.s3.{region}.amazonaws.com/{key}"
    return urls


def deploy(cloudformation, stack_name, parameters):
    body = PARENT_TEMPLATE
    cloudformation.validate_template(TemplateBody=body)
    try:
        current = cloudformation.describe_stacks(StackName=stack_name)["Stacks"][0]
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "ValidationError":
            raise
        current = None
    if current and current["StackStatus"] not in {"CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"}:
        raise RuntimeError(f"Stack {stack_name} is in {current['StackStatus']}; inspect events before retrying")
    supplied = [{"ParameterKey": key, "ParameterValue": str(value)} for key, value in parameters.items()]
    if current:
        existing_keys = {p["ParameterKey"] for p in current["Parameters"]}
        supplied += [{"ParameterKey": key, "UsePreviousValue": True}
                     for key in existing_keys - parameters.keys()]
    args = {
        "StackName": stack_name,
        "TemplateBody": body,
        "Parameters": supplied,
        "Capabilities": ["CAPABILITY_NAMED_IAM"],
        "Tags": [{"Key": "Purpose", "Value": "data-engineering-training"}],
    }
    try:
        if current:
            cloudformation.update_stack(**args)
            waiter = "stack_update_complete"
        else:
            cloudformation.create_stack(**args, OnFailure="ROLLBACK")
            waiter = "stack_create_complete"
        cloudformation.get_waiter(waiter).wait(
            StackName=stack_name, WaiterConfig={"Delay": 30, "MaxAttempts": 100})
    except ClientError as exc:
        if "No updates are to be performed" in exc.response.get("Error", {}).get("Message", ""):
            return cloudformation.describe_stacks(StackName=stack_name)["Stacks"][0]
        raise
    except WaiterError:
        for event in cloudformation.describe_stack_events(StackName=stack_name)["StackEvents"][:20]:
            print(event["LogicalResourceId"], event["ResourceStatus"], event.get("ResourceStatusReason", ""))
        raise
    return cloudformation.describe_stacks(StackName=stack_name)["Stacks"][0]


### Optional: import the existing public key

Run this if the `ec2emrkey` key pair is absent from AWS. It reads the public key from `~/.ssh/ec2emrkey.pem.pub`.


In [ ]:
public_key_path = Path.home() / ".ssh" / "ec2emrkey.pem.pub"
if public_key_path.exists():
    try:
        ec2.import_key_pair(KeyName=KEY_NAME, PublicKeyMaterial=public_key_path.read_bytes())
        print("Imported key:", KEY_NAME)
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "InvalidKeyPair.Duplicate":
            raise
        print("Key already exists:", KEY_NAME)
else:
    print("Public key not found; the key pair must already exist in AWS:", public_key_path)


## 2. Discover the default VPC and allow the lab IP range

The RDS subnet group uses default VPC subnets in distinct Availability Zones. EMR uses one of those subnets. The RDS template creates a dedicated security group for MySQL port 3306. The existing default security group receives the SSH rule for EMR. The notebook also tags the existing EMR subnet and security group as required by the EMR service policy.


In [ ]:
import urllib.request

network = default_network(ec2, KEY_NAME)
public_ip = urllib.request.urlopen("https://checkip.amazonaws.com", timeout=10).read().decode().strip()
allowed_cidr = first_three_octets_cidr(public_ip)
ec2.create_tags(
    Resources=[network["EMRSubnetId"], network["EMRSecurityGroupId"]],
    Tags=[{"Key": "for-use-with-amazon-emr-managed-policies", "Value": "true"}],
)
allow_ssh(ec2, network["EMRSecurityGroupId"], allowed_cidr)
print("Default VPC:", network["VpcId"])
print("RDS default subnets:", network["SubnetIds"])
print("MSK default subnets:", network["MSKSubnetIds"])
print("EMR subnet:", network["EMRSubnetId"])
print("Allowed client range:", allowed_cidr)


## 3. Upload separate templates

CloudFormation nested stacks require S3 URLs. The uploaded key contains a hash of each template's content, so changing only one child template changes only that nested stack URL.


In [ ]:
template_urls = upload_templates(s3, S3_DATALAKE, STACK_NAME, AWS_REGION)
for name, url in template_urls.items():
    print(name, url)


## 4. Deploy IAM, Kinesis, Firehose, and MySQL RDS

The first deployment omits EMR so you can use the other resources while the cluster is still unnecessary. This creates billable AWS resources. The MySQL username is `mysqladmin`. The password comes from `MYSQL_PASSWORD` in the first code cell. CloudFormation marks the password parameter `NoEcho`.


In [ ]:
base_parameters = {
    **template_urls,
    "DeployEMR": "false",
    "DeployMSK": "false",
    "BucketName": S3_DATALAKE,
    "StreamName": KINESIS_STREAM_NAME,
    "FirehoseName": KINESIS_FIREHOSE_NAME,
    "VpcId": network["VpcId"],
    "SubnetIds": ",".join(network["SubnetIds"]),
    "MSKSubnetIds": ",".join(network["MSKSubnetIds"]),
    "VpcCidr": network["VpcCidr"],
    "EMRSubnetId": network["EMRSubnetId"],
    "EMRSecurityGroupId": network["EMRSecurityGroupId"],
    "AllowedCidr": allowed_cidr,
    "DBUsername": MYSQL_USERNAME,
    "DBPassword": MYSQL_PASSWORD,
    "ClusterName": CLUSTER_NAME,
    "KeyName": KEY_NAME,
    "ReleaseLabel": RELEASE_LABEL,
    "InstanceType": INSTANCE_TYPE,
    "CoreInstanceCount": CORE_INSTANCE_COUNT,
    "DataVolumeSizeGB": EBS_DATA_VOLUME_SIZE_GB,
    "RootVolumeSizeGB": EBS_ROOT_VOLUME_SIZE_GB,
    "IdleTimeoutSeconds": IDLE_TIMEOUT_SECONDS,
    "MSKClusterName": MSK_CLUSTER_NAME,
    "MSKBrokerStorageGB": MSK_BROKER_STORAGE_GB,
    "MSKKafkaVersion": MSK_KAFKA_VERSION,
}
try:
    existing_stack = cloudformation.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
except ClientError as exc:
    if exc.response["Error"]["Code"] != "ValidationError":
        raise
else:
    previous = {p["ParameterKey"]: p.get("ParameterValue") for p in existing_stack["Parameters"]}
    base_parameters["DeployEMR"] = previous.get("DeployEMR", "false")
    base_parameters["DeployMSK"] = previous.get("DeployMSK", "false")
stack = deploy(cloudformation, STACK_NAME, base_parameters)
outputs = {item["OutputKey"]: item["OutputValue"] for item in stack.get("Outputs", [])}
print("Stack:", stack["StackStatus"])
print("MySQL:", outputs["RDSEndpoint"], outputs["RDSPort"])
print("Kinesis:", outputs["KinesisStreamName"])
print("Firehose:", outputs["FirehoseDeliveryStreamName"])


### Optional: create Glue databases and crawlers

This keeps the original lab's Glue setup. It creates or updates four databases and crawlers without starting them. The training role has broad permissions, as in the original notebook.


In [ ]:
import json
import time

GLUE_CRAWLERS = {
    "movielens": "bronze/movielens/",
    "olist": "bronze/olist/",
    "ecomm": "bronze/ecomm/",
    "odoo": "bronze/odoo/",
}
TRAINING_ROLE = "GKS_GLUE_EMR_ROLE"
iam = session.client("iam")
glue = session.client("glue")
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
try:
    role = iam.get_role(RoleName=TRAINING_ROLE)["Role"]
except iam.exceptions.NoSuchEntityException:
    role = iam.create_role(
        RoleName=TRAINING_ROLE,
        Path="/service-role/",
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Training Glue crawler role",
    )["Role"]
service_policy_arn = "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"
attached = iam.list_attached_role_policies(RoleName=TRAINING_ROLE)["AttachedPolicies"]
if not any(item["PolicyArn"] == service_policy_arn for item in attached):
    iam.attach_role_policy(RoleName=TRAINING_ROLE, PolicyArn=service_policy_arn)
iam.put_role_policy(
    RoleName=TRAINING_ROLE,
    PolicyName="DataEngTrainingFullAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow", "Action": ["s3:*", "glue:*", "rds:*", "elasticmapreduce:*"], "Resource": "*"}],
    }),
)
for database_name, bronze_prefix in GLUE_CRAWLERS.items():
    s3_path = f"s3://{S3_DATALAKE}/{bronze_prefix.strip('/')}/"
    database_input = {"Name": database_name, "LocationUri": s3_path, "Description": f"Bronze data for {database_name}"}
    try:
        glue.get_database(Name=database_name)
    except glue.exceptions.EntityNotFoundException:
        glue.create_database(DatabaseInput=database_input)
    else:
        glue.update_database(Name=database_name, DatabaseInput=database_input)
    crawler_name = f"{database_name}-bronze-crawler"
    crawler_config = {"Name": crawler_name, "Role": role["Arn"], "DatabaseName": database_name, "Targets": {"S3Targets": [{"Path": s3_path}]}}
    try:
        glue.get_crawler(Name=crawler_name)
    except glue.exceptions.EntityNotFoundException:
        operation = glue.create_crawler
    else:
        operation = glue.update_crawler
    for attempt in range(12):
        try:
            operation(**crawler_config)
            break
        except glue.exceptions.InvalidInputException as exc:
            if "unable to assume provided role" not in str(exc).lower() or attempt == 11:
                raise
            time.sleep(10)
    print(crawler_name, s3_path)


## 5. Deploy EMR when needed

Run this later. It updates the **same parent stack** and adds only the EMR nested stack. The unchanged child template URLs keep the other nested stacks stable. CloudFormation may take many minutes to finish EMR creation.


In [ ]:
emr_parameters = dict(base_parameters)
current_stack = cloudformation.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
current_parameters = {p["ParameterKey"]: p.get("ParameterValue") for p in current_stack["Parameters"]}
emr_parameters["DeployMSK"] = current_parameters.get("DeployMSK", "false")
emr_parameters["DeployEMR"] = "true"
stack = deploy(cloudformation, STACK_NAME, emr_parameters)
outputs = {item["OutputKey"]: item["OutputValue"] for item in stack.get("Outputs", [])}
cluster_id = outputs["EMRClusterId"]
cluster = session.client("emr").describe_cluster(ClusterId=cluster_id)["Cluster"]
print("Stack:", stack["StackStatus"])
print("EMR cluster:", cluster_id, cluster["Status"]["State"])
print("Primary DNS:", cluster.get("MasterPublicDnsName", "not available yet"))


## 6. Deploy provisioned MSK when needed

This final deployment step adds a separate MSK nested stack to the same parent stack. It uses two default VPC subnets in different Availability Zones, one `kafka.t3.small` broker per AZ, and `MSK_BROKER_STORAGE_GB` GiB per broker (default 10). Kafka 3.9.x uses ZooKeeper metadata for this broker type. Clients within the default VPC can use TLS on port 9094. MSK can take a long time to create and incurs broker charges.


In [ ]:
msk_parameters = dict(base_parameters)
current_stack = cloudformation.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
current_parameters = {p["ParameterKey"]: p.get("ParameterValue") for p in current_stack["Parameters"]}
msk_parameters["DeployEMR"] = current_parameters.get("DeployEMR", "false")
msk_parameters["DeployMSK"] = "true"
stack = deploy(cloudformation, STACK_NAME, msk_parameters)
outputs = {item["OutputKey"]: item["OutputValue"] for item in stack.get("Outputs", [])}
msk_arn = outputs["MSKClusterArn"]
brokers = session.client("kafka").get_bootstrap_brokers(ClusterArn=msk_arn)
print("Stack:", stack["StackStatus"])
print("MSK cluster ARN:", msk_arn)
print("TLS bootstrap brokers:", brokers.get("BootstrapBrokerStringTls", "not available yet"))


## 7. Inspect events and clean up

Deleting the parent stack deletes nested stacks, including EMR, RDS, and MSK. Existing S3 data, the S3 bucket, EC2 key pair, and default VPC remain. The SSH rule and EMR resource tags added to existing network resources also remain and can be removed separately.


In [ ]:
for event in cloudformation.describe_stack_events(StackName=STACK_NAME)["StackEvents"][:20]:
    print(event["Timestamp"], event["LogicalResourceId"], event["ResourceStatus"], event.get("ResourceStatusReason", ""))


In [ ]:
# Uncomment when the lab is complete.
# cloudformation.delete_stack(StackName=STACK_NAME)
# cloudformation.get_waiter("stack_delete_complete").wait(
#     StackName=STACK_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 100}
# )
